# `00_environment.ipynb` — Environment, Gemma Architecture & Reproducibility

## 1. Pedagogical Overview & Learning Objectives
In this CAS module, we answer a central engineering question in modern NLP:
> **When is in-context prompting (`0-shot` / `few-shot`) sufficient on an instruction-tuned LLM, and when does parameter-efficient fine-tuning (`LoRA` / `4-bit NF4 QLoRA`) provide decisive gains in accuracy, schema reliability, and inference token efficiency?**

To make this comparison scientifically valid, we hold the **base model checkpoint** (`google/gemma-2-2b-it`), **decoding strategy** (deterministic greedy decoding, `do_sample=False`), **output JSON contract**, and **evaluation metrics** strictly constant across all methods.

### Key Architectural Properties of Google Gemma 2 / Gemma 3 Instruct
1. **Strict Two-Role Turn Structure (`user` / `model` only):**
   Unlike OpenAI or Llama chat templates, Gemma's official Jinja chat template sets `supports_system_role = false`. Calling `tokenizer.apply_chat_template` with `role='system'` raises a `TemplateError`. Every prompt must alternate strictly between `<start_of_turn>user` and `<start_of_turn>model`.
2. **Single Leading `<bos>` Token Rule:**
   Gemma's chat template prepends `<bos>` at the start of the rendered conversation. When tokenizing already-rendered chat strings for generation or `SFTTrainer`, we must avoid adding a second `<bos>` token (`add_special_tokens=False`), which degrades generation quality.
3. **Logit Soft-Capping in Gemma 2 Attention:**
   Gemma 2 applies `tanh` logit soft-capping (`attn_logit_softcapping=50.0`, `final_logit_softcapping=30.0`) to stabilize activations. Using `attn_implementation='eager'` or `'sdpa'` in `transformers` preserves exact soft-capping behavior.

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import Image, display, Markdown
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from germeval_llm.convert import ensure_derived_datasets_ready
ensure_derived_datasets_ready()
print("Repository initialized:", ROOT.name)


In [ ]:
import platform
import yaml
import germeval_llm
from germeval_llm.inference import detect_compute_device
from germeval_llm.official_evaluation import find_java_binary
from germeval_llm.prompts import format_gemma_chat_string, compute_completion_only_labels
from germeval_llm.schemas import ChatMessage
from germeval_llm.reporting import save_json

hw = detect_compute_device()
java_bin = find_java_binary(require_jaxb=False)
model_cfg = yaml.safe_load(Path('configs/model.yaml').read_text())

env_manifest = {
    'python_version': platform.python_version(),
    'platform': platform.platform(),
    'germeval_llm_version': germeval_llm.__version__,
    'hardware': hw,
    'java_binary': java_bin,
    'model_config': model_cfg,
}
save_json(env_manifest, 'results/runs/environment.json')
print(json.dumps(env_manifest, indent=2))

## 2. Inspecting the Gemma Chat Template & Turn Delimiters
Below we inspect the exact character sequence produced by `format_gemma_chat_string()`. Notice how:
- The sequence begins with a single `<bos>`.
- The user instruction and input document are wrapped in `<start_of_turn>user\n...<end_of_turn>\n`.
- The assistant turn starts with `<start_of_turn>model\n`, which serves as our exact boundary for **completion-only loss masking** during LoRA/QLoRA fine-tuning.

In [ ]:
sample_prompt = [ChatMessage(role='user', content='Klassifiziere: Der Zug fällt aus.')]
sample_comp = [ChatMessage(role='assistant', content='{"labels":["Zugfahrt:negative"]}')]
rendered = format_gemma_chat_string(sample_prompt, sample_comp)
print('Rendered Gemma sequence:\n' + rendered)
assert rendered.count('<bos>') == 1
assert '<start_of_turn>model\n' in rendered

### Hands-on Classroom Exercise
Inspect what happens when a `system` message is passed to `format_gemma_chat_string`: verify that it is merged cleanly into the `user` turn rather than producing an invalid `<start_of_turn>system` tag.

In [ ]:
sys_and_user = [
    ChatMessage(role='system', content='Antworte nur mit validem JSON.'),
    ChatMessage(role='user', content='Text: WLAN im ICE geht nicht.'),
]
merged_str = format_gemma_chat_string(sys_and_user)
print(merged_str)
assert '<start_of_turn>system' not in merged_str

### Key Takeaways
- Our hardware and software manifest is logged in `results/runs/environment.json`.
- Gemma Instruct models require `user` and `model` turn delimiters (`<start_of_turn>user` / `<start_of_turn>model`) with a single `<bos>` token.
- All canonical and derived JSONL datasets are automatically materialized from the repository's compact `data/canonical/*.jsonl.gz` archives—no manual XML preprocessing is required on Google Colab.